# Sultan Hassan SDXL LoRA Training — BULLETPROOF EDITION
This notebook uses the FIXED xformers/fp16 setup AND mounts your Google Drive. Every 100 steps, your AI is permanently saved to your personal Google Drive. If Colab ever kicks you out again, your checkpoints are 100% safe, and it will AUTO-RESUME from where you left off!

In [ ]:
# Step 0: Mount Google Drive (Crucial for saving your LoRA permanently!)
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Step 1: Install required dependencies (Using the stable v0.30.3 stack and matching transformers version)
!pip install diffusers==0.30.3 transformers==4.44.2 accelerate peft datasets wandb xformers

In [ ]:
# Step 2: Upload your curated dataset
import os
import zipfile
from google.colab import files

print("Please upload your zipped dataset (data_final.zip):")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

!rm -rf dataset
!unzip -q -o {zip_name} -d dataset
print(f"Dataset unzipped to ./dataset/! Found {len(os.listdir('dataset'))} images.")


In [ ]:
# Step 3: Download the official v0.30.3 training script
!wget -q https://raw.githubusercontent.com/huggingface/diffusers/v0.30.3/examples/dreambooth/train_dreambooth_lora_sdxl.py
print("Stable script downloaded.")


In [ ]:
# Step 4: Configure accelerate non-interactively
from accelerate.utils import write_basic_config
write_basic_config(mixed_precision="fp16")
print("accelerate config written.")


In [ ]:
# Step 5: Start Training! (Auto-Resumes if disconnected!)
!accelerate launch --mixed_precision="fp16" train_dreambooth_lora_sdxl.py \
  --pretrained_model_name_or_path="stabilityai/stable-diffusion-xl-base-1.0" \
  --pretrained_vae_model_name_or_path="madebyollin/sdxl-vae-fp16-fix" \
  --instance_data_dir="dataset" \
  --instance_prompt="sltnhsn" \
  --output_dir="/content/drive/MyDrive/sultan_hassan_sdxl_lora" \
  --resolution=1024 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --learning_rate=1e-4 \
  --lr_scheduler="constant" \
  --lr_warmup_steps=0 \
  --max_train_steps=1000 \
  --checkpointing_steps=100 \
  --resume_from_checkpoint="latest" \
  --seed=42 \
  --gradient_checkpointing \
  --enable_xformers_memory_efficient_attention


In [ ]:
# Step 6: Generate beautiful test images right here in Colab!
from diffusers import DiffusionPipeline, AutoencoderKL
import torch
from IPython.display import display

vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)
pipe = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    vae=vae,
    torch_dtype=torch.float16,
    variant="fp16",
).to("cuda")

# We load the LoRA directly from your Google Drive now!
pipe.load_lora_weights("/content/drive/MyDrive/sultan_hassan_sdxl_lora")

prompt = "A beautiful professional photograph of sltnhsn at sunset, highly detailed, dramatic lighting"
image = pipe(prompt, num_inference_steps=30).images[0]

image.save("sultan_hassan_test.png")
display(image)
